# Notebook 1: Data Cleaning + Merging for Global Data Science Project
This notebook handles the cleaning and merging of five global datasets following best practices and the PACE workflow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

RAW_DATA_PATH = 'data/raw/'
PROCESSED_DATA_PATH = 'data/processed/'
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

In [ ]:
# Load Raw Datasets
mortality = pd.read_csv(os.path.join(RAW_DATA_PATH, 'mortality.csv'))
gdp = pd.read_csv(os.path.join(RAW_DATA_PATH, 'worldbank_gdp.csv'))
health_exp = pd.read_csv(os.path.join(RAW_DATA_PATH, 'worldbank_health_exp.csv'))
fao = pd.read_excel(os.path.join(RAW_DATA_PATH, 'fao_food_price_index.xlsx'))
acled = pd.read_csv(os.path.join(RAW_DATA_PATH, 'acled_country_year.csv'))

## Initial Inspection
Check first few rows, info, describe, missing values, and duplicates for each dataset.

In [ ]:
for df, name in zip([mortality, gdp, health_exp, fao, acled], ['Mortality', 'GDP', 'Health Expenditure', 'FAO Food Price Index', 'ACLED']):
    print(f'--- {name} ---')
    display(df.head())
    print(df.info())
    print(df.describe())
    print('Missing values:\n', df.isnull().sum())
    print('Duplicates:', df.duplicated().sum())

## Cleaning Steps
Standardize columns, convert years, handle missing values, harmonize country names, aggregate FAO and ACLED data.

In [ ]:
def clean_columns(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

mortality = clean_columns(mortality)
gdp = clean_columns(gdp)
health_exp = clean_columns(health_exp)
fao = clean_columns(fao)
acled = clean_columns(acled)

for df in [mortality, gdp, health_exp, fao, acled]:
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce')

mortality = mortality.dropna(subset=['country', 'year', 'mortality_rate'])
gdp['gdp_per_capita'].fillna(gdp['gdp_per_capita'].median(), inplace=True)
health_exp['health_expenditure'].fillna(health_exp['health_expenditure'].median(), inplace=True)
fao = fao.dropna(subset=['country', 'year', 'food_price_index'])
acled = acled.fillna(0)

country_corrections = {'United States of America': 'United States', 'Russian Federation': 'Russia'}
for df in [mortality, gdp, health_exp, fao, acled]:
    df['country'] = df['country'].replace(country_corrections)

if 'month' in fao.columns:
    fao_yearly = fao.groupby(['country', 'year'])['food_price_index'].mean().reset_index()
else:
    fao_yearly = fao.copy()

acled_agg = acled.groupby(['country', 'year']).sum().reset_index()

## Merging Datasets
Merge all datasets on country and year using inner joins to keep only common entries.

In [ ]:
merged = mortality.merge(gdp, on=['country','year'], how='inner')
merged = merged.merge(health_exp, on=['country','year'], how='inner')
merged = merged.merge(fao_yearly, on=['country','year'], how='inner')
merged = merged.merge(acled_agg, on=['country','year'], how='inner')
merged.to_csv(os.path.join(PROCESSED_DATA_PATH, 'merged_global_data.csv'), index=False)

## Basic Checks and Visualization
Inspect merged data shape, missing values, correlation, and histograms.

In [ ]:
print('Merged data shape:', merged.shape)
print('Missing values in merged data:\n', merged.isnull().sum())

numeric_cols = merged.select_dtypes(include=np.number).columns
sns.heatmap(merged[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

merged[numeric_cols].hist(bins=20, figsize=(15,10))
plt.tight_layout()
plt.show()

## Save Intermediate Processed Files
Store processed individual datasets for reproducibility.

In [ ]:
for df, name in zip([mortality, gdp, health_exp, fao_yearly, acled_agg], ['mortality', 'gdp', 'health_exp', 'fao', 'acled']):
    df.to_csv(os.path.join(PROCESSED_DATA_PATH, f'processed_{name}.csv'), index=False)